# HW12: временные ряды, temporal split, baseline-модели и GRU

Ноутбук соответствует требованиям HW12 и содержит 4 явных эксперимента: **B1**, **B2**, **B3**, **R1**.


## 1) Импорты, seed и среда


In [ ]:
import json
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
WINDOW_SIZE = 24
HORIZON = 1
EPOCHS = 25
BATCH_SIZE = 64
LR = 1e-3
DATASET_PATH = 'Data/S12-hw-dataset.csv'
ARTIFACTS_DIR = 'artifacts'
FIGURES_DIR = os.path.join(ARTIFACTS_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def mape(y_true, y_pred):
    eps = 1e-8
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100.0)


def metrics_dict(y_true, y_pred):
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mape': mape(y_true, y_pred),
    }


class SeqDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


class GRURegressor(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def make_sequences(series, window_size):
    x, y = [], []
    for i in range(window_size, len(series)):
        x.append(series[i - window_size:i].reshape(-1, 1))
        y.append(series[i])
    return np.asarray(x), np.asarray(y)


set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


## 2) Данные и первичный анализ


In [ ]:
df = pd.read_csv(DATASET_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print('Размер:', df.shape)
print('Диапазон дат:', df['date'].min(), '->', df['date'].max())
print('Пропуски по колонкам:')
print(df.isna().sum())

plt.figure(figsize=(14, 4))
plt.plot(df['date'], df['target'])
plt.title('Исходный временной ряд target')
plt.tight_layout()
plt.show()


## 3) Корректный temporal split


In [ ]:
n = len(df)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print('train:', train_df['date'].min(), '->', train_df['date'].max(), len(train_df))
print('val:  ', val_df['date'].min(), '->', val_df['date'].max(), len(val_df))
print('test: ', test_df['date'].min(), '->', test_df['date'].max(), len(test_df))

plt.figure(figsize=(14, 4))
plt.plot(train_df['date'], train_df['target'], label='train')
plt.plot(val_df['date'], val_df['target'], label='validation')
plt.plot(test_df['date'], test_df['target'], label='test')
plt.legend()
plt.title('Temporal split')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'series_split.png'), dpi=150)
plt.show()

print('\nПочему random split некорректен:')
print('- Он смешивает прошлое и будущее между train/validation/test.')
print('- Это даёт утечку данных и завышенные метрики прогноза.')


### Комментарий про утечки данных

- Лаговые и rolling-признаки строятся через `shift(1)`, чтобы не использовать текущее/будущее значение таргета.
- `StandardScaler` обучается только на train-части и затем применяется к validation/test.
- Лучшая модель выбирается по `validation`, а `test` используется один раз для финальной оценки.


## 4) Лаговые, rolling и календарные признаки


In [ ]:
feat = df.copy()
feat['lag_1'] = feat['target'].shift(1)
feat['lag_7'] = feat['target'].shift(7)
feat['lag_14'] = feat['target'].shift(14)
feat['rolling_mean_7'] = feat['target'].shift(1).rolling(7).mean()
feat['rolling_std_7'] = feat['target'].shift(1).rolling(7).std()
feat['day_of_week'] = feat['date'].dt.dayofweek
feat['hour'] = feat['date'].dt.hour
feat = feat.dropna().reset_index(drop=True)
feat.head()


## 5) Эксперименты B1/B2/B3/R1

Ниже идут **четыре отдельные секции** с явными метриками на validation.


In [ ]:
# ????? ?????????? ??? ?????????????
n_total = len(df)
split_train_end = train_end
split_val_end = val_end

# baseline series
baseline_df = df.copy()
baseline_df['naive_last'] = baseline_df['target'].shift(1)
baseline_df['moving_avg_24'] = baseline_df['target'].shift(1).rolling(24).mean()

# features for B3
feat_full = df.copy()
feat_full['lag_1'] = feat_full['target'].shift(1)
feat_full['lag_7'] = feat_full['target'].shift(7)
feat_full['lag_14'] = feat_full['target'].shift(14)
feat_full['rolling_mean_7'] = feat_full['target'].shift(1).rolling(7).mean()
feat_full['rolling_std_7'] = feat_full['target'].shift(1).rolling(7).std()
feat_full['day_of_week'] = feat_full['date'].dt.dayofweek
feat_full['hour'] = feat_full['date'].dt.hour
feat_full = feat_full.dropna().reset_index(drop=True)

feature_cols = ['lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7', 'day_of_week', 'hour']
train_cut_date = train_df['date'].iloc[-1]
val_cut_date = val_df['date'].iloc[-1]

feat_train = feat_full[feat_full['date'] <= train_cut_date]
feat_val = feat_full[(feat_full['date'] > train_cut_date) & (feat_full['date'] <= val_cut_date)]
feat_test = feat_full[feat_full['date'] > val_cut_date]


### B1: `naive-last`


In [ ]:
val_b = baseline_df.iloc[split_train_end:split_val_end].dropna().copy()
b1_val_metrics = metrics_dict(val_b['target'].values, val_b['naive_last'].values)
print('B1 validation metrics:', {k: round(v, 4) for k, v in b1_val_metrics.items()})


### B2: `moving-average`


In [ ]:
val_b = baseline_df.iloc[split_train_end:split_val_end].dropna().copy()
b2_val_metrics = metrics_dict(val_b['target'].values, val_b['moving_avg_24'].values)
print('B2 validation metrics:', {k: round(v, 4) for k, v in b2_val_metrics.items()})


### B3: `ridge-lag-features`


In [ ]:
x_train = feat_train[feature_cols].values
y_train = feat_train['target'].values
x_val = feat_val[feature_cols].values
y_val = feat_val['target'].values

scaler_x = StandardScaler()
x_train_scaled = scaler_x.fit_transform(x_train)
x_val_scaled = scaler_x.transform(x_val)

ridge = Ridge(alpha=1.0, random_state=SEED)
ridge.fit(x_train_scaled, y_train)
ridge_val_pred = ridge.predict(x_val_scaled)
b3_val_metrics = metrics_dict(y_val, ridge_val_pred)
print('B3 validation metrics:', {k: round(v, 4) for k, v in b3_val_metrics.items()})


### R1: `gru-forecast`


In [ ]:
set_seed(SEED)

y_train_full = train_df['target'].values.astype(np.float32)
y_val_full = val_df['target'].values.astype(np.float32)
y_test_full = test_df['target'].values.astype(np.float32)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train_full.reshape(-1, 1)).reshape(-1)
y_val_scaled = scaler_y.transform(y_val_full.reshape(-1, 1)).reshape(-1)
y_test_scaled = scaler_y.transform(y_test_full.reshape(-1, 1)).reshape(-1)

train_x, train_y = make_sequences(y_train_scaled, WINDOW_SIZE)
val_input = np.concatenate([y_train_scaled[-WINDOW_SIZE:], y_val_scaled])
val_x, val_y = make_sequences(val_input, WINDOW_SIZE)

test_hist = np.concatenate([y_train_scaled, y_val_scaled])[-WINDOW_SIZE:]
test_input = np.concatenate([test_hist, y_test_scaled])
test_x, test_y = make_sequences(test_input, WINDOW_SIZE)

train_loader = DataLoader(SeqDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SeqDataset(val_x, val_y), batch_size=BATCH_SIZE, shuffle=False)

model = GRURegressor().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history_train_loss = []
history_val_mae = []
best_val_mae = float('inf')
best_state = None

for _ in range(EPOCHS):
    model.train()
    running_loss = 0.0
    n_batches = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        n_batches += 1
    history_train_loss.append(running_loss / max(n_batches, 1))

    model.eval()
    val_preds_scaled = []
    val_true_scaled = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            preds = model(xb).cpu().numpy()
            val_preds_scaled.append(preds)
            val_true_scaled.append(yb.numpy())

    val_preds_scaled = np.concatenate(val_preds_scaled)
    val_true_scaled = np.concatenate(val_true_scaled)
    val_preds = scaler_y.inverse_transform(val_preds_scaled.reshape(-1, 1)).reshape(-1)
    val_true = scaler_y.inverse_transform(val_true_scaled.reshape(-1, 1)).reshape(-1)
    val_mae_epoch = mean_absolute_error(val_true, val_preds)
    history_val_mae.append(float(val_mae_epoch))

    if val_mae_epoch < best_val_mae:
        best_val_mae = float(val_mae_epoch)
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state is None:
    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

torch.save(best_state, os.path.join(ARTIFACTS_DIR, 'best_gru.pt'))
model.load_state_dict(best_state)
model.eval()

with torch.no_grad():
    val_preds_scaled = model(torch.tensor(val_x, dtype=torch.float32).to(device)).cpu().numpy()
val_preds = scaler_y.inverse_transform(val_preds_scaled.reshape(-1, 1)).reshape(-1)
val_true = scaler_y.inverse_transform(val_y.reshape(-1, 1)).reshape(-1)
r1_val_metrics = metrics_dict(val_true, val_preds)

print('R1 validation metrics:', {k: round(v, 4) for k, v in r1_val_metrics.items()})


## 6) Выбор лучшей модели по validation и одна финальная оценка на test


In [ ]:
val_scores = {
    'B1': b1_val_metrics['mae'],
    'B2': b2_val_metrics['mae'],
    'B3': b3_val_metrics['mae'],
    'R1': r1_val_metrics['mae'],
}

print('Validation MAE по экспериментам:')
for k, v in val_scores.items():
    print(f'{k}: {v:.4f}')

best_exp = min(val_scores, key=val_scores.get)
print('Лучшая модель по val MAE:', best_exp)

test_metrics = {'B1': None, 'B2': None, 'B3': None, 'R1': None}
best_test_true = None
best_test_pred = None

if best_exp == 'B1':
    t = baseline_df.iloc[split_val_end:].dropna().copy()
    best_test_true = t['target'].values
    best_test_pred = t['naive_last'].values
    test_metrics['B1'] = metrics_dict(best_test_true, best_test_pred)
elif best_exp == 'B2':
    t = baseline_df.iloc[split_val_end:].dropna().copy()
    best_test_true = t['target'].values
    best_test_pred = t['moving_avg_24'].values
    test_metrics['B2'] = metrics_dict(best_test_true, best_test_pred)
elif best_exp == 'B3':
    x_test = feat_test[feature_cols].values
    y_test = feat_test['target'].values
    best_test_true = y_test
    best_test_pred = ridge.predict(scaler_x.transform(x_test))
    test_metrics['B3'] = metrics_dict(best_test_true, best_test_pred)
else:
    with torch.no_grad():
        preds_test_scaled = model(torch.tensor(test_x, dtype=torch.float32).to(device)).cpu().numpy()
    best_test_pred = scaler_y.inverse_transform(preds_test_scaled.reshape(-1, 1)).reshape(-1)
    best_test_true = scaler_y.inverse_transform(test_y.reshape(-1, 1)).reshape(-1)
    test_metrics['R1'] = metrics_dict(best_test_true, best_test_pred)

print('Финальные test-метрики (только для выбранной лучшей):', test_metrics[best_exp])


## 7) Сохранение артефактов и таблицы `runs.csv`


In [ ]:
# figures
plt.figure(figsize=(10, 4))
exps = ['B1', 'B2', 'B3', 'R1']
vals = [val_scores[e] for e in exps]
plt.bar(exps, vals)
plt.title('Validation MAE comparison')
plt.ylabel('MAE')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'baselines_compare.png'), dpi=150)
plt.close()

plt.figure(figsize=(10, 4))
plt.plot(history_train_loss, label='train_loss')
plt.plot(history_val_mae, label='val_mae')
plt.title('GRU learning curves')
plt.xlabel('epoch')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'gru_learning_curves.png'), dpi=150)
plt.close()

plt.figure(figsize=(14, 4))
plt.plot(best_test_true, label='actual')
plt.plot(best_test_pred, label='prediction')
plt.title(f'Best model test forecast ({best_exp})')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'best_forecast_test.png'), dpi=150)
plt.close()

config = {
    'seed': SEED,
    'window_size': WINDOW_SIZE,
    'horizon': HORIZON,
    'hidden_size': 32,
    'num_layers': 1,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'optimizer': 'Adam',
    'lr': LR,
    'scaler': 'StandardScaler(target)',
    'device': str(device),
}
with open(os.path.join(ARTIFACTS_DIR, 'best_gru_config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)


def row(exp_id, model_summary, features_summary, scaler, optimizer_name, lr_value, epochs_trained, val_m):
    tm = test_metrics[exp_id]
    return {
        'experiment_id': exp_id,
        'task': 'forecasting',
        'dataset': DATASET_PATH,
        'seed': SEED,
        'split_summary': f'train=0:{split_train_end}, val={split_train_end}:{split_val_end}, test={split_val_end}:{n_total}',
        'window_size': WINDOW_SIZE if exp_id == 'R1' else '',
        'horizon': HORIZON,
        'model_summary': model_summary,
        'features_summary': features_summary,
        'scaler': scaler,
        'optimizer': optimizer_name,
        'lr': lr_value,
        'epochs_trained': epochs_trained,
        'best_val_mae': val_m['mae'],
        'best_val_rmse': val_m['rmse'],
        'best_val_mape': val_m['mape'],
        'test_mae': '' if tm is None else tm['mae'],
        'test_rmse': '' if tm is None else tm['rmse'],
        'test_mape': '' if tm is None else tm['mape'],
        'notes': 'test evaluated only for selected best-by-validation experiment' if tm is not None else '',
    }


runs_rows = [
    row('B1', 'naive-last', 'last value', 'none', 'none', '', 0, b1_val_metrics),
    row('B2', 'moving-average', 'rolling mean(24)', 'none', 'none', '', 0, b2_val_metrics),
    row('B3', 'ridge', 'lag_1,lag_7,lag_14,rolling_mean_7,rolling_std_7,day_of_week,hour', 'StandardScaler(X)', 'closed_form', '', 1, b3_val_metrics),
    row('R1', 'GRU(input=1,hidden=32,layers=1)', 'windowed target series', 'StandardScaler(target)', 'Adam', LR, EPOCHS, r1_val_metrics),
]
runs_df = pd.DataFrame(runs_rows)
runs_df.to_csv(os.path.join(ARTIFACTS_DIR, 'runs.csv'), index=False)
print('runs.csv saved')


In [ ]:
runs = pd.read_csv('artifacts/runs.csv')
runs


In [ ]:
best_idx = runs['best_val_mae'].idxmin()
best_exp = runs.loc[best_idx, 'experiment_id']
print('Лучшая модель по validation MAE:', best_exp)
print('Финальная оценка на test выполнялась только для выбранной лучшей модели.')
